# FD001 Sample Weight Experiments

Objetivo: priorizar casos cercanos a falla con pesos de entrenamiento y medir si baja el error peligroso sin degradar demasiado el error global. Solo validación artificial.


In [ ]:
from collections import OrderedDict
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RESULTS_DIR = PROJECT_ROOT / "results"
FIGURES_DIR = PROJECT_ROOT / "figures"
PREDICTIONS_DIR = PROJECT_ROOT / "predictions"
for path in [RESULTS_DIR, FIGURES_DIR, PREDICTIONS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

SEED = 42
DATA_DIR = PROJECT_ROOT / "CMAPSSData"
EVAL_SIZE = 0.2
CUT_RULS = (20, 50, 80, 110, 140)
DEFAULT_WINDOW_SIZE = 30
DEFAULT_RUL_CAP = 125

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 140)


In [ ]:
from src.preprocessed_FD001 import (
    prepare_fd001_temporal_validation_only,
    preprocessing_summary,
)
from src.fd001_modeling import (
    metrics_by_model,
    metrics_by_rul_bin,
    plot_validation_diagnostics,
    prediction_frame,
    regression_metrics,
    set_global_seed,
)


def add_bin_metric_columns(metrics, bin_metrics):
    result = metrics.copy()
    for label in ["0-30", "30-60", "60-90", "90+"]:
        safe_label = label.replace("-", "_").replace("+", "plus")
        subset = bin_metrics.loc[bin_metrics["rul_bin"].astype(str) == label]
        for _, row in subset.iterrows():
            mask = (
                (result["representation"] == row["representation"])
                & (result["model_name"] == row["model_name"])
            )
            result.loc[mask, f"mae_rul_{safe_label}"] = row["mae"]
            result.loc[mask, f"dangerous_error_pct_rul_{safe_label}"] = row["dangerous_error_pct"]
    return result


def available_tabular_factories(random_state=42):
    from sklearn.ensemble import (
        ExtraTreesRegressor,
        GradientBoostingRegressor,
        HistGradientBoostingRegressor,
        RandomForestRegressor,
    )

    factories = OrderedDict()
    availability = []
    has_external_boosting = False

    factories["RandomForestRegressor"] = lambda: RandomForestRegressor(
        n_estimators=250,
        max_depth=14,
        min_samples_leaf=3,
        random_state=random_state,
        n_jobs=-1,
    )

    try:
        from xgboost import XGBRegressor
        factories["XGBRegressor"] = lambda: XGBRegressor(
            n_estimators=160,
            max_depth=3,
            learning_rate=0.04,
            subsample=0.85,
            colsample_bytree=0.85,
            objective="reg:squarederror",
            random_state=random_state,
            n_jobs=-1,
            tree_method="hist",
            verbosity=0,
        )
        has_external_boosting = True
        availability.append("XGBoost disponible: se incluye XGBRegressor.")
    except Exception as exc:
        availability.append(f"XGBoost no disponible: {type(exc).__name__}.")

    try:
        from lightgbm import LGBMRegressor
        factories["LGBMRegressor"] = lambda: LGBMRegressor(
            n_estimators=220,
            max_depth=-1,
            num_leaves=31,
            learning_rate=0.035,
            subsample=0.85,
            colsample_bytree=0.85,
            reg_lambda=0.1,
            random_state=random_state,
            n_jobs=-1,
            verbose=-1,
        )
        has_external_boosting = True
        availability.append("LightGBM disponible: se incluye LGBMRegressor.")
    except Exception as exc:
        availability.append(f"LightGBM no disponible: {type(exc).__name__}.")

    if not has_external_boosting:
        availability.append("Sin XGBoost/LightGBM: se usan fallbacks sklearn.")
        factories["HistGradientBoostingRegressor"] = lambda: HistGradientBoostingRegressor(
            max_iter=180,
            learning_rate=0.05,
            max_leaf_nodes=31,
            l2_regularization=0.01,
            random_state=random_state,
        )
        factories["GradientBoostingRegressor"] = lambda: GradientBoostingRegressor(
            n_estimators=160,
            learning_rate=0.05,
            max_depth=3,
            subsample=0.85,
            random_state=random_state,
        )
        factories["ExtraTreesRegressor"] = lambda: ExtraTreesRegressor(
            n_estimators=220,
            max_depth=16,
            min_samples_leaf=2,
            random_state=random_state,
            n_jobs=-1,
        )
    return factories, availability

def fit_predict_model(prepared, model_name, model, representation, sample_weight=None):
    if sample_weight is None:
        model.fit(prepared["X_train"], prepared["y_train"])
    else:
        model.fit(prepared["X_train"], prepared["y_train"], sample_weight=sample_weight)
    preds = model.predict(prepared["X_eval"])
    return prediction_frame(
        prepared["eval_df"],
        preds,
        model_name=model_name,
        representation=representation,
    ), model


In [ ]:
def near_failure_weights(y_raw):
    y_raw = np.asarray(y_raw, dtype=float)
    return np.select(
        [y_raw <= 30, (y_raw > 30) & (y_raw <= 60), (y_raw > 60) & (y_raw <= 90)],
        [4.0, 2.0, 1.5],
        default=1.0,
    )


In [ ]:
set_global_seed(SEED)
prepared = prepare_fd001_temporal_validation_only(
    data_dir=DATA_DIR,
    eval_size=EVAL_SIZE,
    random_state=SEED,
    max_rul=DEFAULT_RUL_CAP,
    cut_ruls=CUT_RULS,
    window_size=DEFAULT_WINDOW_SIZE,
)
weights = near_failure_weights(prepared["y_train_raw"])
pd.Series(weights).value_counts().sort_index()


In [ ]:
model_factories, availability_notes = available_tabular_factories(random_state=SEED)
boosting_metrics_path = RESULTS_DIR / "fd001_boosting_metrics.csv"
if boosting_metrics_path.exists():
    previous = pd.read_csv(boosting_metrics_path)
    candidates = previous.loc[previous["model_name"] != "RandomForestRegressor"]
    best_boosting_name = candidates.sort_values(["rmse", "mae"]).iloc[0]["model_name"] if not candidates.empty else "HistGradientBoostingRegressor"
else:
    best_boosting_name = "XGBRegressor" if "XGBRegressor" in model_factories else "HistGradientBoostingRegressor"

models_to_compare = OrderedDict([
    ("RandomForestRegressor", model_factories["RandomForestRegressor"]),
    (best_boosting_name, model_factories[best_boosting_name]),
])
models_to_compare.keys()


## Comparación sin pesos vs con pesos


In [ ]:
representation = f"temporal_w{DEFAULT_WINDOW_SIZE}"
prediction_tables = []

for base_name, factory in models_to_compare.items():
    for use_weights in [False, True]:
        model_name = f"{base_name}_{'weighted' if use_weights else 'unweighted'}"
        print(f"Entrenando {model_name}")
        model = factory()
        fit_weights = weights if use_weights else None
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            pred_df, _ = fit_predict_model(prepared, model_name, model, representation, sample_weight=fit_weights)
        prediction_tables.append(pred_df)

predictions = pd.concat(prediction_tables, ignore_index=True)
metrics = metrics_by_model(predictions)
bin_metrics = metrics_by_rul_bin(predictions)
metrics = add_bin_metric_columns(metrics, bin_metrics)
metrics.insert(2, "window_size", DEFAULT_WINDOW_SIZE)
metrics.insert(3, "rul_cap", DEFAULT_RUL_CAP)
metrics.insert(4, "n_features", len(prepared["feature_columns"]))
metrics.insert(5, "sample_weight_policy", "near_failure_priority")
metrics


In [ ]:
metrics.to_csv(RESULTS_DIR / "fd001_sample_weight_metrics.csv", index=False)
predictions.to_csv(RESULTS_DIR / "fd001_sample_weight_predictions.csv", index=False)
bin_metrics.to_csv(RESULTS_DIR / "fd001_sample_weight_metrics_by_rul_bin.csv", index=False)

display(metrics.sort_values(["dangerous_error_pct", "rmse"]))
display(bin_metrics.sort_values(["rul_bin", "dangerous_error_pct", "mae"]))
print(RESULTS_DIR / "fd001_sample_weight_metrics.csv")
